# Prepare HAM10000 Dataset

Thin Colab runner for Sprint 1 dataset audit and split generation. Core logic lives in `src/dl_midterm/` and `scripts/`.

Skip this notebook when you want to preserve the uploaded Sprint 1 split CSVs. Use it only when intentionally re-running the dataset audit/split step.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/KasimDeliaci/dl-midterm-project.git'
REPO_ROOT = Path('/content/dl-assignment')
BUNDLE_CANDIDATES = [
    Path('/content/drive/MyDrive/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar'),
]
DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/dl-midterm-artifacts')

drive.mount('/content/drive', force_remount=True)
DRIVE_BUNDLE = next((path for path in BUNDLE_CANDIDATES if path.exists()), BUNDLE_CANDIDATES[0])

if not (REPO_ROOT / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())
print('Drive bundle exists:', DRIVE_BUNDLE.exists(), DRIVE_BUNDLE)
print('Checked bundle candidates:')
for path in BUNDLE_CANDIDATES:
    print(' -', path, path.exists())

required_paths = [
    REPO_ROOT / 'data/raw/ham10000',
    REPO_ROOT / 'data/metadata/HAM10000_metadata.csv',
    REPO_ROOT / 'data/splits/train.csv',
    REPO_ROOT / 'data/splits/val.csv',
    REPO_ROOT / 'data/splits/test.csv',
]

if not all(path.exists() for path in required_paths):
    if not DRIVE_BUNDLE.exists():
        raise FileNotFoundError(f'Missing Drive bundle: {DRIVE_BUNDLE}')
    subprocess.run(['tar', '--warning=no-unknown-keyword', '-xf', str(DRIVE_BUNDLE), '-C', str(REPO_ROOT)], check=True)

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError(f'Missing required dataset paths after extraction: {missing}')

print('Dataset and preserved Sprint 1 split CSVs are available.')
for path in required_paths:
    print(path.relative_to(REPO_ROOT))

In [ ]:
!python -m pip install -q uv
!uv sync

In [ ]:
RUN_DATASET_PREPARE = False

if RUN_DATASET_PREPARE:
    import subprocess
    subprocess.run(['uv', 'run', 'python', 'scripts/prepare_dataset.py', '--config', 'configs/dataset/selected_dataset.yaml'], check=True)
else:
    print('Skipped. Set RUN_DATASET_PREPARE = True only if you intentionally want to regenerate audit/split outputs.')